# Can a Computer Taste? 🍪🤖
**Splash at Yale — E5492**

Today you are the data. You tasted four foods and rated each one on **sweet**, **salty**, **sour**, and **how enjoyable** it was.

Our question: **if a computer only knows how sweet, salty, and sour a food is, can it predict how much you'll enjoy it?**

👉 Run each cell with **Shift + Enter**. Only change code where it says ✏️.

## Step 0 — Setup
Loads the tools and the class data. (Your teacher fills in the link.)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Link to the class responses (published Google Sheet as CSV)
TASTE_URL  = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQTBQLZG6f3yxOUXnvs6mx9elfPfl9f2dD_epdzEecIKO-sMcOLiRqfhkKKE7LZJ_XvH4IrCa6X7x0R/pub?gid=1907066911&single=true&output=csv"
WASABI_URL = None   # if wasabi is in a separate form, paste its CSV link here; None = look in TASTE_URL

USE_FAKE_DATA = False   # True = practice data (use if the real data isn't ready)

TRAIN_FOODS = ["Pringles", "Pocky", "Sour candy", "Crackers"]
TEST_FOOD   = "Wasabi Peas"
TASTES      = ["sweet", "salty", "sour"]   # what the computer gets to see
TARGET      = "enjoyable"                  # what it tries to predict

## Step 1 — Load the data
Each row of the spreadsheet is one person. We reshape it so **each row is one person tasting one food**.

In [ ]:
def to_long(wide):
    """Turn 'Food | rating' columns into rows: student, food, sweet, salty, sour, enjoyable."""
    wide = wide.copy()
    wide.columns = [str(c).strip() for c in wide.columns]
    wide["student"] = range(1, len(wide) + 1)
    rows = []
    for col in wide.columns:
        if "|" not in col:
            continue
        food, rating = [p.strip() for p in col.split("|", 1)]
        part = pd.DataFrame({"student": wide["student"], "food": food,
                             "rating": rating.lower(),
                             "value": pd.to_numeric(wide[col], errors="coerce")})
        rows.append(part)
    if not rows:
        return pd.DataFrame(columns=["student", "food"] + TASTES + [TARGET])
    long = pd.concat(rows).pivot_table(index=["student", "food"], columns="rating",
                                       values="value").reset_index()
    long.columns.name = None
    long = long.reindex(columns=["student", "food"] + TASTES + [TARGET])
    return long.dropna(subset=TASTES + [TARGET])

def fake_data(n=20, seed=0):
    """Made-up ratings that look roughly like a real class."""
    rng = np.random.default_rng(seed)
    #               sweet salty sour  enjoyable
    profile = {"Pringles":    (2, 8, 1, 8), "Pocky":    (8, 2, 1, 8),
               "Sour candy":  (7, 1, 8, 7), "Crackers": (5, 3, 1, 6),
               "Wasabi Peas": (1, 7, 1, 4)}
    rows = []
    for s in range(1, n + 1):
        for food, (sw, sa, so, en) in profile.items():
            r = lambda m: int(np.clip(round(m + rng.normal(0, 1.5)), 1, 10))
            rows.append({"student": s, "food": food, "sweet": r(sw), "salty": r(sa),
                         "sour": r(so), "enjoyable": r(en + rng.normal(0, 1.5))})
    return pd.DataFrame(rows)

data = None
if not USE_FAKE_DATA:
    try:
        data = to_long(pd.read_csv(TASTE_URL))
        if WASABI_URL:
            data = pd.concat([data, to_long(pd.read_csv(WASABI_URL))], ignore_index=True)
        if (data["food"].isin(TRAIN_FOODS)).sum() == 0:
            print("⚠️  No responses yet — using practice data for now.")
            data = None
    except Exception as e:
        print("⚠️  Couldn't load the class data (", type(e).__name__, ") — using practice data.")
        data = None
if data is None:
    data = fake_data()
    print("📋 Using PRACTICE data.")
else:
    print("✅ Loaded REAL class data!")

train  = data[data["food"].isin(TRAIN_FOODS)].reset_index(drop=True)
wasabi = data[data["food"] == TEST_FOOD].reset_index(drop=True)
print(f"{train['student'].nunique()} tasters, {len(train)} training rows, {len(wasabi)} wasabi rows")
train.head(8)

## Step 2 — Look at the data
What does the class think, on average?

In [ ]:
train.groupby("food")[TASTES + [TARGET]].mean().round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, taste in zip(axes, TASTES):
    for food in TRAIN_FOODS:
        d = train[train["food"] == food]
        jitter = np.random.uniform(-0.2, 0.2, len(d))
        ax.scatter(d[taste] + jitter, d[TARGET] + jitter, label=food, alpha=0.7)
    ax.set_xlabel(f"how {taste} (1–10)")
    ax.set_xlim(0, 11); ax.set_ylim(0, 11)
axes[0].set_ylabel("how enjoyable (1–10)")
axes[0].legend()
plt.suptitle("Each dot is one person tasting one food")
plt.tight_layout(); plt.show()

🤔 **Talk with your partner:** Does saltier = more enjoyable? Sweeter? More sour? Do people agree with each other?

## Step 3 — You vs. the computer: guess the weights
Our model is a recipe:

$$\text{enjoyment} = \text{start} + w_{\text{sweet}}\cdot\text{sweet} + w_{\text{salty}}\cdot\text{salty} + w_{\text{sour}}\cdot\text{sour}$$

A **weight** says how much that taste matters. Positive = makes food better, negative = makes it worse, 0 = doesn't matter.

✏️ **Change the numbers below and re-run. Try to get the error as low as you can!**

In [ ]:
# ✏️ YOUR GUESSES
start    = 3.0
w_sweet  = 0.3
w_salty  = 0.3
w_sour   = 0.0

guess = start + w_sweet*train["sweet"] + w_salty*train["salty"] + w_sour*train["sour"]
my_error = (guess - train[TARGET]).abs().mean()
print(f"Your predictions are off by {my_error:.2f} points on average (out of 10).")

## Step 4 — Let the computer train
**Training** = the computer tries weights over and over to find the ones with the smallest error. Did you beat it?

In [ ]:
model = LinearRegression()
model.fit(train[TASTES], train[TARGET])

computer_error = (model.predict(train[TASTES]) - train[TARGET]).abs().mean()
print(f"Computer's start: {model.intercept_:.2f}")
for taste, w in zip(TASTES, model.coef_):
    print(f"  weight for {taste:>5}: {w:+.2f}   (each extra point of {taste} changes enjoyment by {w:+.2f})")
print(f"\nComputer's error: {computer_error:.2f}    Your error: {my_error:.2f}")
print("🏆 You beat the computer?!" if my_error < computer_error else "🤖 The computer wins this round.")

## Step 5 — The challenge: wasabi peas 🟢
✍️ **Before running:** write down your guess — how enjoyable will the class find wasabi peas?

The computer only sees how **sweet, salty, and sour** the wasabi peas are. Let's see what it predicts…

In [ ]:
if len(wasabi) == 0:
    print("No wasabi ratings yet — submit the wasabi form, then re-run Step 1 and this cell.")
else:
    wasabi = wasabi.copy()
    wasabi["predicted"] = model.predict(wasabi[TASTES]).clip(1, 10)
    print("Average wasabi taste ratings:")
    print(wasabi[TASTES].mean().round(1).to_string())
    print(f"\n🤖 Computer predicts enjoyment: {wasabi['predicted'].mean():.1f} / 10")

👅 **Now everyone eats a wasabi pea.** Then run the reveal:

In [ ]:
if len(wasabi):
    actual, predicted = wasabi[TARGET].mean(), wasabi["predicted"].mean()
    print(f"🤖 Predicted: {predicted:.1f}    😋 Actual: {actual:.1f}    Off by: {abs(actual - predicted):.1f}")

    train_avg = train.groupby("food")[TARGET].mean()
    names = list(train_avg.index) + ["Wasabi\n(predicted)", "Wasabi\n(actual)"]
    vals  = list(train_avg.values) + [predicted, actual]
    colors = ["#9aa5b1"] * len(train_avg) + ["#4c78a8", "#e45756"]
    plt.figure(figsize=(8, 4))
    plt.bar(names, vals, color=colors)
    plt.ylim(0, 10); plt.ylabel("average enjoyment")
    plt.title("How did the computer do on a food it never tasted?")
    plt.show()

🤔 **Talk with your partner:** Was the computer right? What does wasabi have that the computer can't "see"?

## Step 6 — Fix it: add a "spicy" column?
Wasabi is **spicy**. Let's give the computer a spiciness column and retrain.

✏️ Set how spicy you think wasabi peas are. (Pringles, Pocky, sour candy and crackers aren't spicy, so they get 0.)

In [ ]:
wasabi_spicy = 8   # ✏️ 1–10

train2 = train.copy();  train2["spicy"] = 0
model2 = LinearRegression().fit(train2[TASTES + ["spicy"]], train2[TARGET])
print(f"Weight the computer learned for spicy: {model2.coef_[-1]:+.2f}")

if len(wasabi):
    w2 = wasabi.copy(); w2["spicy"] = wasabi_spicy
    pred2 = model2.predict(w2[TASTES + ["spicy"]]).clip(1, 10).mean()
    print(f"New prediction for wasabi: {pred2:.1f}   (before: {wasabi['predicted'].mean():.1f}, actual: {wasabi[TARGET].mean():.1f})")

😮 **The spicy weight is 0 — nothing changed!** Every food the computer trained on had spiciness = 0, so it never saw what spiciness *does*.

**Big idea:** a computer can only learn from what's in its data. Adding a new feature (column) doesn't help unless the training data has *examples* of it. More complex models need more data.

## 🧠 Discussion
1. What did the model get right? What did it get wrong?
2. Is "how sweet" easier to predict than "how enjoyable"? Why might people agree on one more than the other?
3. Besides spiciness, what else affects taste that we never measured? (Hint: pinch your nose 👃)
4. What data would we need for the computer to understand spicy food?
5. **Can a computer taste?**

### 🚀 Bonus (if you finish early)
Change `TASTES` in Step 0 to only `["salty"]` or `["sweet"]`, re-run everything, and see how the error changes. Which taste matters most?